# 05 · Sesgo-varianza con función verdadera conocida

**Módulo 3 · Sesión 8** — Evaluación y selección de modelos

## Objetivos

Sobre datos sintéticos —donde se conoce la función que generó los datos, algo imposible con
datos reales— este notebook:

1. Ajusta polinomios de grado creciente y **mide** el patrón en U del error de validación
   que predice `05-sesgo-varianza-validacion.md`.
2. Muestra que el "mejor grado" elegido con un solo split cambia según la semilla del split
   — la motivación exacta de la validación cruzada.
3. Implementa k-fold **a mano** y compara su curva de validación, con barras de error, contra
   la de un solo split.
4. Construye curvas de aprendizaje para un modelo con sesgo alto, uno razonable y uno con
   varianza alta, y verifica que más datos ayuda a unos y no a otros.

**Paquetes:** `numpy`, `pandas`, `matplotlib`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-3-regresion-evaluacion/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Datos con función verdadera conocida

$y = \sin(2\pi x) + \varepsilon$, con $x$ uniforme en $[0,1]$ y $\varepsilon \sim
\mathcal{N}(0, 0.3^2)$. Ningún modelo puede bajar el error esperado de $0.3^2=0.09$ — ese es
el ruido irreducible de `05-sesgo-varianza-validacion.md`.

In [ ]:
n = 120
x = np.sort(rng.uniform(0, 1, n))
f_verdadera = lambda x: np.sin(2 * np.pi * x)
y = f_verdadera(x) + rng.normal(0, 0.3, n)

fig, eje = plt.subplots(figsize=(7, 4))
eje.scatter(x, y, alpha=0.5, s=20, label="datos observados")
malla = np.linspace(0, 1, 200)
eje.plot(malla, f_verdadera(malla), color="black", linewidth=2, label="función verdadera")
eje.legend()
plt.tight_layout()
plt.show()

## 2. Regresión polinómica a mano, de grado variable

Se reutiliza la idea de `01-regresion-lineal.md`: una matriz de diseño con las potencias de
$x$ sigue siendo un modelo lineal en $\boldsymbol{\beta}$. Se usa mínimos cuadrados vía SVD
(`lstsq`) en vez de la ecuación normal directa: a partir de grado 10 la matriz
$\mathbf{X}^\top\mathbf{X}$ se vuelve casi singular (`03-multicolinealidad-polinomica.md`) y
`lstsq` es la variante numéricamente estable de la misma solución.

In [ ]:
def matriz_polinomica(x, grado):
    return np.column_stack([x**d for d in range(grado + 1)])


def ajustar_ols(X, y):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta


def mse(y, y_pred):
    return np.mean((y - y_pred) ** 2)

## 3. Un solo split: la curva en U, pero ruidosa

Se ajustan grados de 1 a 15, se mide el error en un único split 70/30, y se repite con seis
semillas de split distintas para ver qué tan estable es "el mejor grado".

In [ ]:
grados = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
n_train = int(0.7 * n)

mejores_por_semilla = []
for semilla_split in range(6):
    rng_split = np.random.default_rng(200 + semilla_split)
    idx = rng_split.permutation(n)
    tr, va = idx[:n_train], idx[n_train:]
    errores_val = []
    for grado in grados:
        Xd = matriz_polinomica(x, grado)
        beta = ajustar_ols(Xd[tr], y[tr])
        errores_val.append(mse(y[va], Xd[va] @ beta))
    mejores_por_semilla.append(grados[int(np.argmin(errores_val))])

pd.DataFrame({"semilla_del_split": range(6), "mejor_grado": mejores_por_semilla})

El "mejor grado" cambia con cada semilla del split, sin ningún patrón —el mismo problema
que `02-proyecto-reproducible-aplicado.ipynb` (módulo 1) advirtió sobre confiar en una sola
partición—. No se puede recomendar un grado con esta evidencia sola.

## 4. K-fold a mano: de un número a una distribución

Se implementa `k_fold_indices` desde cero —`scikit-learn` tiene `KFold`, pero el mecanismo
es tan simple que vale la pena verlo una vez sin la envoltura de la librería—.

In [ ]:
def k_fold_indices(n, k, rng):
    idx = rng.permutation(n)
    return np.array_split(idx, k)


K = 5
folds = k_fold_indices(n, K, np.random.default_rng(7))

errores_por_grado = {}  # se guardan los K errores, no solo su resumen: la sección 4.1 los usa
resultados_kfold = []
for grado in grados:
    Xd = matriz_polinomica(x, grado)
    errores = []
    for i in range(K):
        va_idx = folds[i]
        tr_idx = np.concatenate([folds[j] for j in range(K) if j != i])
        beta = ajustar_ols(Xd[tr_idx], y[tr_idx])
        errores.append(mse(y[va_idx], Xd[va_idx] @ beta))
    errores_por_grado[grado] = np.array(errores)
    resultados_kfold.append(
        {
            "grado": grado,
            "media": np.mean(errores),
            "ee": np.std(errores) / np.sqrt(K),
        }
    )
resultados_kfold = pd.DataFrame(resultados_kfold)
resultados_kfold.round(4)

In [ ]:
fig, eje = plt.subplots(figsize=(8, 4.5))
eje.errorbar(
    resultados_kfold["grado"],
    resultados_kfold["media"],
    yerr=resultados_kfold["ee"],
    marker="o",
    capsize=3,
)
eje.axhline(0.09, color="black", linestyle="--", linewidth=1, label="ruido irreducible ($\\sigma^2=0.09$)")
eje.set_xlabel("Grado del polinomio")
eje.set_ylabel("MSE de validación (media ± ee, 5 pliegues)")
eje.set_title("Curva de validación con barras de error")
eje.legend()
plt.tight_layout()
plt.show()

La forma en U aparece con claridad, y ahora con una medida de cuánto confiar en cada punto.
Los grados 3 a 8 tienen medias muy parecidas entre sí, dentro de una o dos barras de error.
Del grado 9 en adelante el error medio sube **y** la barra de error se ensancha
notablemente: no solo empeora, se vuelve más errático entre pliegues — la firma de entrar en
el régimen de alta varianza.

### 4.1 Barras que se solapan **no** son todavía una comparación pareada

Es tentador mirar dos barras de error que se solapan y concluir "no hay diferencia". No es lo
mismo que la comparación pareada de `05-sesgo-varianza-validacion.md`, sección 3, y conviene
ver por qué. Cada barra describe **un** grado por separado, y buena parte de su ancho no
viene del grado sino de que unos pliegues son intrínsecamente más difíciles que otros — algo
que afecta a los dos modelos por igual. La comparación pareada resta pliegue a pliegue y
cancela justamente esa parte común.

Aquí se puede hacer de verdad, porque los 12 grados se evaluaron sobre **los mismos** 5
pliegues.

In [ ]:
GRADO_A, GRADO_B = 5, 7
d = errores_por_grado[GRADO_A] - errores_por_grado[GRADO_B]
ee_pareado = d.std() / np.sqrt(K)

print(f"MSE medio    — grado {GRADO_A}: {errores_por_grado[GRADO_A].mean():.4f}"
      f"   grado {GRADO_B}: {errores_por_grado[GRADO_B].mean():.4f}")
print(f"ee por separado — grado {GRADO_A}: {errores_por_grado[GRADO_A].std()/np.sqrt(K):.4f}"
      f"   grado {GRADO_B}: {errores_por_grado[GRADO_B].std()/np.sqrt(K):.4f}")
print(f"\nDiferencia media pareada ({GRADO_A} − {GRADO_B}): {d.mean():+.4f}")
print(f"Error estándar de la diferencia:        {ee_pareado:.4f}")
print(f"|diferencia| / ee = {abs(d.mean())/ee_pareado:.2f}  (regla práctica: hace falta > 2)")

Dos lecturas, y la segunda es la que importa:

1. **La conclusión no cambia:** la diferencia entre el grado 5 y el 7 es 1.2 errores
   estándar, por debajo de la regla de 2 — indistinguible del ruido de muestreo, igual que
   sugerían las barras solapadas.
2. **Pero la medición es mucho más fina:** el error estándar de la *diferencia* (0.0035) es
   menor que el de cualquiera de los dos grados por separado (0.0093 y 0.0065). Emparejar no
   es una formalidad — al cancelar la dificultad común de cada pliegue, detecta diferencias
   que las barras individuales, más anchas, esconderían. El notebook 06 y el ejercicio 03
   usan exactamente esta herramienta sobre Ames Housing.

## 5. Curvas de aprendizaje: ¿ayuda tener más datos?

Se eligen tres grados representativos y se mide el error de entrenamiento y de validación en
función del tamaño de la muestra de entrenamiento, promediando 30 repeticiones por tamaño
para suavizar el ruido de cuál subconjunto exacto se usa.

In [ ]:
idx_completo = rng.permutation(n)
pool_entrenamiento = idx_completo[:90]
validacion_fija = idx_completo[90:]
tamanos = np.arange(15, 91, 5)

curvas = {}
for grado in [1, 5, 15]:
    Xd = matriz_polinomica(x, grado)
    medias_tr, medias_va = [], []
    for m in tamanos:
        errores_tr, errores_va = [], []
        for rep in range(30):
            rng_rep = np.random.default_rng(1000 + rep)
            sub = rng_rep.choice(pool_entrenamiento, size=m, replace=False)
            beta = ajustar_ols(Xd[sub], y[sub])
            errores_tr.append(mse(y[sub], Xd[sub] @ beta))
            errores_va.append(mse(y[validacion_fija], Xd[validacion_fija] @ beta))
        medias_tr.append(np.mean(errores_tr))
        medias_va.append(np.mean(errores_va))
    curvas[grado] = (np.array(medias_tr), np.array(medias_va))

In [ ]:
etiquetas = {1: "grado 1 (subajuste)", 5: "grado 5 (razonable)", 15: "grado 15 (sobreajuste)"}
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2))

for eje, grado in zip(ejes, [1, 5, 15]):
    tr, va = curvas[grado]
    eje.plot(tamanos, tr, marker="o", markersize=3, label="entrenamiento")
    eje.plot(tamanos, va, marker="o", markersize=3, label="validación")
    if grado == 15:
        eje.set_yscale("log")
    eje.set_xlabel("Tamaño de la muestra de entrenamiento")
    eje.set_ylabel("MSE")
    eje.set_title(etiquetas[grado])
    eje.legend(fontsize=8)

plt.tight_layout()
plt.show()

Los tres patrones de `05-sesgo-varianza-validacion.md`, uno por panel:

- **Grado 1 (subajuste):** entrenamiento y validación convergen, pero a un error alto
  (≈0.22-0.25) — muy por encima del ruido irreducible (0.09). Más datos no ayudan: el
  modelo es demasiado simple para la curva verdadera, sin importar cuántos puntos se le den.
- **Grado 5 (razonable):** la brecha entre entrenamiento y validación es pequeña y se cierra
  con más datos; el error de validación converge cerca del ruido irreducible.
- **Grado 15 (sobreajuste):** con muestras chicas, el error de validación es
  *astronómicamente* alto (nótese la escala logarítmica: pasa de más de $10^{14}$ con 15
  puntos a valores razonables con 90) — un polinomio de grado 15 con pocos puntos está
  groseramente subdeterminado. La brecha con el entrenamiento se va cerrando a medida que
  crece la muestra, y hacia $m=90$ el error de validación se acerca al del grado 5. La
  varianza sí se cura con más datos; el sesgo del grado 1, no.

## Resumen

| Lo que se midió | Conecta con |
|---|---|
| "Mejor grado" inestable entre semillas de un solo split | `05-sesgo-varianza-validacion.md`, sección 2 |
| K-fold con barras de error: grados 3-8 indistinguibles entre sí, ≥9 claramente peor y más errático | Curva de validación, `05-sesgo-varianza-validacion.md` sección 4 |
| Comparación pareada grado 5 vs. 7: misma conclusión, pero con un ee menor que el de cada grado por separado | `05-sesgo-varianza-validacion.md` sección 3; notebook 06 y ejercicio 03 |
| Curva de aprendizaje del grado 1: sesgo alto, más datos no ayudan | Diagnóstico de sesgo vs. varianza |
| Curva de aprendizaje del grado 15: varianza altísima con poca muestra, se cura con más datos | Motiva por qué el notebook 06 usa CV en vez de un solo split para elegir hiperparámetros |